In [12]:
import pandas as pd
import re
import time

from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException

In [44]:
# Testowane na szerokości 1110 pikseli
download_service = Service()
driver = webdriver.Chrome(service=download_service)

sklep_opon_base_url = "https://www.sklepopon.com/szukaj-opony?sezon=zimowe&rozmiar=205/55R16&ofs="
oponeo_base_url = "https://www.oponeo.pl/wybierz-opony/s=1/zimowe/t=1/osobowe/r=1/205-55-r16"

In [14]:
def close_sklep_opon_popups(outer_driver):
    try:
        btn_cookie = outer_driver.find_element(By.CSS_SELECTOR, "#klaro > div > div > div > div > div > button")
        btn_cookie.click()
        print("Przycisk akceptacji ciasteczek został kliknięty.")
    except NoSuchElementException:
        print("Przycisk akceptacji ciasteczek nie został znaleziony.")
    except Exception as exception:
        print("Nie udało się kliknąć przycisku akceptacji ciasteczek:", exception)
    
    try:
        outer_driver.execute_script("""
            const shadowHost = document.querySelector("body > div.gr-visual-prompt");
            if (shadowHost) {
                const shadowRoot = shadowHost.shadowRoot;
                const closeButton = shadowRoot.querySelector("div > div:nth-child(2) > button:nth-child(1)");
                if (closeButton) {
                    closeButton.click();
                    console.log("Okienko powiadomień zostało zamknięte.");
                } else {
                    console.log("Nie znaleziono przycisku zamknięcia powiadomień.");
                }
            } else {
                console.log("Okno powiadomień nie zostało znalezione.");
            }
        """)
    except Exception as exception:
        print("Nie udało się zamknąć okienka powiadomień:", exception)

def close_oponeo_popup(outer_driver):
    try:
        reject_button = outer_driver.find_element(By.CSS_SELECTOR, "#consentsBar > div.buttonsContainer.container > div > span.reject")
        reject_button.click()
        print("Okienko prywatności zostało zamknięte.")
    except NoSuchElementException:
        print("Okienko prywatności nie jest widoczne lub zostało już zamknięte.")
    except Exception as e:
        print("Wystąpił błąd podczas zamykania okienka prywatności:", e)
        
def load_sklep_opon_tire_data(outer_driver):
    scrapped_data = []
    try:
        opony_elements = outer_driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')
            
        class_mapping = {
            "Premium": "Premium",
            "Średnia": "Średnia",
            "Średniej": "Średnia",
            "Ekonomiczna": "Ekonomiczna",
            "Ekonomicznej": "Ekonomiczna"
        }
        
        for opona_element in opony_elements:
            
            try:
                load_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="li"]').text
                speed_index = opona_element.find_element(By.CSS_SELECTOR, 'li[data-attribute-code="si"]').text
            except NoSuchElementException:
                load_index = None
                speed_index = None
            
            noise_level = None
            try:
                noise_level_elements = opona_element.find_elements(By.CSS_SELECTOR, "span.self-center.tracking-tighter.sm\\:tracking-normal")
                for noise_level_element in noise_level_elements:
                    text = noise_level_element.text
                    match = re.search(r'\d+', text)
                    if match:
                        noise_level = int(match.group())
            except (NoSuchElementException, IndexError):
                pass
            
            etykieta_elements = opona_element.find_elements(By.CSS_SELECTOR, 'span.icon-fuel-new ~ span, span.icon-rain-new ~ span, span.icon-speaker-new ~ span')
            fuel_index = etykieta_elements[0].text if len(etykieta_elements) > 0 else None
            wet_grip_index = etykieta_elements[1].text if len(etykieta_elements) > 1 else None
            noise_index = etykieta_elements[2].text.split(" ")[0] if len(etykieta_elements) > 2 else None
            
            try:
                tire_class_element = opona_element.find_element(By.XPATH, ".//*[contains(translate(text(), 'ABCDEFGHIJKLMNOPQRSTUVWXYZ', 'abcdefghijklmnopqrstuvwxyz'), 'klas')]")
                tire_class_text = tire_class_element.text.lower().replace("w klasie ", "").replace("klasa ", "").strip().capitalize()
                tire_class = class_mapping.get(tire_class_text, tire_class_text)
            except NoSuchElementException:
                tire_class = None
                
            try:
                user_rating_element = opona_element.find_element(By.XPATH, ".//li[contains(@class, 'xl:hidden')]//span[contains(@class, 'ml-1')]")
                user_rating = float(user_rating_element.text.replace(",", "."))
            except NoSuchElementException:
                user_rating = None
                
            price = float(opona_element.get_attribute('data-ee-product-properties').split(";")[3].split(":")[1])
                
            tire_data = {
                "name": opona_element.get_attribute("data-ee-product-properties").split(";")[0].split(":")[1],
                "brand": opona_element.get_attribute("data-ee-product-properties").split(";")[4].split(":")[1],
                "model": opona_element.get_attribute("data-ee-product-properties").split(";")[6].split(":")[1],
                "size": opona_element.get_attribute("data-ee-product-properties").split(";")[5].split(":")[1],
                "load_index": load_index,
                "speed_index": speed_index,
                "fuel_index": fuel_index,
                "wet_grip_index": wet_grip_index,
                "noise_index": noise_index,
                "noise_level": noise_level,
                "class": tire_class,
                "user_rating": user_rating,
                "price": price,
            }
            
            scrapped_data.append(tire_data)
    finally:
        pass
    return scrapped_data   

def load_next_oponeo_page(web_driver, current_page_number):
    try:
        next_page_button = web_driver.find_element(By.ID, f"_ctPgrp_pi{current_page_number}i")
        next_page_button.click()
        time.sleep(2)
        return True
    except NoSuchElementException:
        return False
    
def load_oponeo_tire_data(outer_driver):
    scrapped_data = []
    products = outer_driver.find_elements(By.CLASS_NAME, "product")
    
    for product in products:
        try:
            try:
                link_element = product.find_element(By.CSS_SELECTOR, ".productName a")
                nazwa = link_element.get_attribute("title")
            except NoSuchElementException:
                nazwa = product.find_element(By.CLASS_NAME, "productName").text

            noise = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text
            match = re.search(r'\d+', noise)
            if match:
                noise_level = int(match.group())
            else:
                noise_level = int(product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[1].replace("dB", "").strip())
             
            noise_index = product.find_element(By.CSS_SELECTOR, ".icon-noise em").text.split()[0] 
            noise_index_text = noise_index if noise_index in {"A", "B", "C", "D", "E", "F"} and len(noise_index) == 1 else None   
                
            try:
                user_rating = product.find_element(By.CSS_SELECTOR, ".productRating .note").text
            except NoSuchElementException:
                user_rating = None
                
            tire_info = {
                "name": nazwa,
                "brand": product.find_element(By.CLASS_NAME, "producerName").text,
                "model": product.find_element(By.CLASS_NAME, "modelName").text,
                "size": product.find_element(By.CLASS_NAME, "modelSize").text,
                "load_index": product.find_element(By.XPATH, ".//span[@data-tp='TireLoadIndex']/em").text,
                "speed_index": product.find_element(By.XPATH, ".//span[@data-tp='TireSpeedIndex']/em").text,
                "fuel_index": product.find_element(By.CSS_SELECTOR, ".icon-fuel em").text,
                "wet_grip_index": product.find_element(By.CSS_SELECTOR, ".icon-rain em").text,
                "noise_index": noise_index_text,
                "noise_level": noise_level,
                "class": product.find_element(By.CLASS_NAME, "class").text.replace("KLASA ", "").capitalize(),
                "user_rating": user_rating,
                "price": product.find_element(By.CLASS_NAME, "priceValue").text,
            }
            
            scrapped_data.append(tire_info)

        except NoSuchElementException:
            pass
            
    return scrapped_data    

In [45]:
# Kod do pobierania danych ze strony sklep opon
sklep_opon_tires_data = []
offset = 0

while True:
    url = f"{sklep_opon_base_url}{offset}"
    driver.get(url)
    time.sleep(4)
    
    if offset == 0:
        close_sklep_opon_popups(driver)
    
    tires_data = load_sklep_opon_tire_data(driver)
    sklep_opon_tires_data.extend(tires_data)
    
    offset += 20
    if len(driver.find_elements(By.CSS_SELECTOR, 'div[data-c-name="listing-products-element"]')) == 0:
        print("Brak nowych danych. Koniec paginacji.")
        break

df_sklep_opon = pd.DataFrame(sklep_opon_tires_data)
display(df_sklep_opon)

Przycisk akceptacji ciasteczek został kliknięty.
Brak nowych danych. Koniec paginacji.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price
0,BluEarth Winter V906 205/55 R16 91 T,Yokohama,BluEarth Winter V906,205/55 R16,91,T,D,B,B,71.0,Premium,5.3,347.00
1,Wintercraft WP52 205/55 R16 91 H,Kumho,Wintercraft WP52,205/55 R16,91,H,C,B,B,72.0,Średnia,5.1,320.00
2,Frigo HP2 205/55 R16 91 H,Dębica,Frigo HP2,205/55 R16,91,H,C,C,B,72.0,Ekonomiczna,5.2,283.00
3,Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71.0,Ekonomiczna,5.1,239.00
4,Winter i*cept RS3 W462 205/55 R16 91 T,Hankook,Winter i*cept RS3 W462,205/55 R16,91,T,C,B,B,72.0,Premium,5.3,322.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...
406,I FIT+ 205/55 R16 91 T,Laufenn,I FIT+,205/55 R16,91,T,C,C,B,72.0,None,5.4,324.28
407,UltraGrip 9+ 205/55 R16 94 H,Goodyear,UltraGrip 9+,205/55 R16,94,H,C,B,B,71.0,Premium,5.5,558.64
408,Blizzak LM005 205/55 R16 94 V,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71.0,Premium,5.5,637.89
409,Winter Sottozero Serie II 205/55 R16 94 V,Pirelli,Winter Sottozero Serie II,205/55 R16,94,V,C,C,B,72.0,Premium,5.5,1059.57


In [46]:
# Kod do pobierania danych ze strony oponeo
driver.get(oponeo_base_url)
close_oponeo_popup(driver)

all_tires_data = []

page_number = 1
while True:
    tires_data = load_oponeo_tire_data(driver)
    all_tires_data.extend(tires_data)
    
    page_number += 1
    if not load_next_oponeo_page(driver, page_number):
        break

df_oponeo = pd.DataFrame(all_tires_data)
display(df_oponeo)

Okienko prywatności zostało zamknięte.


,name,brand,model,size,load_index,speed_index,fuel_index,wet_grip_index,noise_index,noise_level,class,user_rating,price
0,Bridgestone Blizzak LM005 205/55 R16 91 H,Bridgestone,Blizzak LM005,205/55 R16,91,H,C,A,B,71,Premium,"4,7",459
1,Michelin Alpin 7 205/55 R16 91 H,Michelin,Alpin 7,205/55 R16,91,H,C,B,B,71,Premium,"4,8",467
2,Dębica Frigo 2 205/55 R16 91 T,Dębica,Frigo 2,205/55 R16,91,T,C,C,B,71,Ekonomiczna,"4,2",252
3,Kormoran Snow 205/55 R16 91 H,Kormoran,Snow,205/55 R16,91,H,D,C,B,72,Ekonomiczna,"4,5",249
4,Firemax FM805+ 205/55 R16 91 H,Firemax,FM805+,205/55 R16,91,H,D,C,A,67,Ekonomiczna,"4,4",209
...,...,...,...,...,...,...,...,...,...,...,...,...,...
219,Goodyear UG Performance 2 205/55 R16 91 H RUN ...,Goodyear,UG Performance 2,205/55 R16,91,H,D,C,B,72,Premium,"4,3",778
220,Bridgestone Blizzak LM005 205/55 R16 94 V XL,Bridgestone,Blizzak LM005,205/55 R16,94,V,C,A,B,71,Premium,"4,7",854
221,Barum Polaris 5 205/55 R16 94 V XL,Barum,Polaris 5,205/55 R16,94,V,C,C,B,72,Ekonomiczna,"4,4",854
222,Pirelli SottoZero Serie 3 205/55 R16 91 H RUN ...,Pirelli,SottoZero Serie 3,205/55 R16,91,H,D,B,B,72,Premium,"4,6",972


In [47]:
# Zamknięcie drivera po scrapowaniu danych
driver.quit()
# opcjonalnie zapis danych do plików CSV jak ktoś potrzebuje
# df_sklep_opon.to_csv("data/miniprojekt/sklep_opon.csv", index=False)
# df_oponeo.to_csv("data/miniprojekt/oponeo.csv", index=False)

In [55]:
# Weryfikacja poprawnych wartości w każdej kolumnie
print("sklep opon")
print(df_sklep_opon.dtypes)
# for column in df_sklep_opon.columns:
#     if column not in ["name", "brand", "model"]:
#         print(column, df_sklep_opon[column].unique())
print("oponeo")
print(df_oponeo.dtypes)
# for column in df_oponeo.columns:
#     if column not in ["name", "brand", "model"]:
#         print(column, df_oponeo[column].unique())

sklep opon
name               object
brand              object
model              object
size               object
load_index         object
speed_index        object
fuel_index         object
wet_grip_index     object
noise_index        object
noise_level         Int64
class              object
user_rating       float64
price             float64
dtype: object
oponeo
name               object
brand              object
model              object
size               object
load_index         object
speed_index        object
fuel_index         object
wet_grip_index     object
noise_index        object
noise_level         Int64
class              object
user_rating       float64
price             float64
dtype: object


In [56]:
# Czyszczenie i przygotowanie danych

# Konwersja kolumn do typów numerycznych w zbiorze df_oponeo
df_oponeo['noise_level'] = pd.to_numeric(df_oponeo['noise_level'], errors='coerce').astype('Int64')
df_oponeo['user_rating'] = pd.to_numeric(df_oponeo['user_rating'], errors='coerce').astype(float)
df_oponeo['price'] = pd.to_numeric(df_oponeo['price'], errors='coerce').astype(float)

# Konwersja kolumn do typów numerycznych w zbiorze df_sklep_opon
df_sklep_opon['noise_level'] = pd.to_numeric(df_sklep_opon['noise_level'], errors='coerce').astype('Int64')
df_sklep_opon['user_rating'] = pd.to_numeric(df_sklep_opon['user_rating'], errors='coerce').astype(float)
df_sklep_opon['price'] = pd.to_numeric(df_sklep_opon['price'], errors='coerce').astype(float)

# 1. Uzupełnianie noise_index na podstawie noise_level
# Średnie wartości noise_index dla każdej wartości noise_level
grouped_noise_index_oponeo = df_oponeo.groupby('noise_index')['noise_level'].mean()
grouped_noise_index_oponeo = grouped_noise_index_oponeo.round().astype(int)

if 'C' not in grouped_noise_index_oponeo:
    if 'B' in grouped_noise_index_oponeo and 'A' in grouped_noise_index_oponeo:
        grouped_noise_index_oponeo['C'] = grouped_noise_index_oponeo['B'] + grouped_noise_index_oponeo['B'] - grouped_noise_index_oponeo['A']
        
noise_map_oponeo = grouped_noise_index_oponeo.to_dict()


# Funkcja do znalezienia najbliższej wartości w słowniku
def find_closest_match(value, noise_map):
    # Obliczamy różnicę między wartością i każdą wartością w słowniku, a potem wybieramy najmniejszą różnicę
    closest_key = min(noise_map, key=lambda k: abs(noise_map[k] - value))
    return closest_key

# Krok 2: Uaktualniamy kolumnę 'noise_index' w dataframe
def update_noise_index(row, noise_map):
    if pd.notna(row['noise_level']) and pd.isna(row['noise_index']):
        closest_key = find_closest_match(row['noise_level'], noise_map)
        return closest_key
    return row['noise_index']

# Zastosowanie funkcji do dataframe
df_oponeo['noise_index'] = df_oponeo.apply(update_noise_index, axis=1, noise_map=noise_map_oponeo)

print("Zaktualizowano noise_index w zbiorze df_oponeo")
print(df_oponeo['noise_index'].unique())

Zaktualizowano noise_index w zbiorze df_oponeo
['B' 'A' 'C']
